# ಪಾಠ 08 - ಬಹು ಏಜೆಂಟ್ ವಿನ್ಯಾಸ ಮಾದರಿ


## ಹೊಂದಿಕೆ


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ಬಹು-ಏಜೆಂಟ್ ವ್ಯವಸ್ಥೆಗಳು ಯಾಕೆ?

ಪ್ರಯಾಣ ಯೋಜನೆ ಮಾಡುವಂತಹ ನೈಜ ಜಗತ್ತಿನ ಕಾರ್ಯಗಳು ಅನೇಕ ವಿಧದ ಪರಿಣಿತರ ತೊಡಕನ್ನು ಒಳಗೊಂಡಿವೆ — ಸಾಗಣೆ, ಸ್ಥಳೀಯ ಜ್ಞಾನ, ಬಜೆಟಿಂಗ್ ಮತ್ತು ಮತ್ತಷ್ಟು. ಎಲ್ಲವನ್ನೂ ನಿರ್ವಹಿಸುವ ಪ್ರಯತ್ನ ಮಾಡುವ ಒಬ್ಬ ಏಜೆಂಟ್ ಶೀಘ್ರದಲ್ಲಿಯೇ ಅಸಹ್ಯವಾಗುವುದು.

ಬಹು-ಏಜೆಂಟ್ ವ್ಯವಸ್ಥೆಗಳು ಇದನ್ನು **ನಿಪುಣತೆಯಿಂದ** ಪರಿಹರಿಸುತ್ತವೆ: ಪ್ರತಿಯೊಂದು ಏಜೆಂಟ್ ಒಂದು ಕ್ಷೇತ್ರದ ಪರಿಣತಿಯನ್ನು ಗಮನಿಸುತ್ತದೆ, ಸಾಮಾನ್ಯ ವ್ಯಕ್ತಿಗಿಂತ ಹೆಚ್ಚಿನ ಗುಣಮಟ್ಟದ ಫಲಿತಾಂಶಗಳನ್ನು ಉತ್ಪಾದಿಸುತ್ತದೆ. ಅವು **ಪ್ರವಣತೆಯನ್ನು** ಕೂಡ ಹೆಚ್ಚಿಸಬಹುದು — ನೀವು ಹೊಸ ಏಜೆಂಟ್‌ಗಳನ್ನು (ಉದಾ. ವಿಮಾನ ತಜ್ಞ, ರೆಸ್ಟೋರೆಂಟ್ ವಿಮರ್ಶಕ) ಈ ಹೊಂದಾಣಿಕೆಯ ಕೆಲಸದ ಪ್ರಕ್ರಿಯೆಯನ್ನು ಮರುಬರೆಯದೆ ಸೇರಿಸಬಹುದು. ಏಜೆಂಟ್‌ಗಳು ಸಂರಚಿತ ಪೈಪ್ಲೈನ್ ಮೂಲಕ ಸಂಯೋಜിക്കಲ್ಪಡುತ್ತವೆ, ಒಂದುೕತರದಿಂದ ಎರಡಕ್ಕೆ ಸಂದರ್ಭವನ್ನು ಹಸ್ತಾಂತರಿಸಿ.


## ವಿಶೇಷ ಏಜೆಂಟ್ಗಳ ಸೃಷ್ಟಿ


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## ಕ್ರಮಬದ್ಧ ಕಾರ್ಯಪ್ರವಾಹ ನಿರ್ಮಾಣ

`WorkflowBuilder` ನಿಮಗೆ ಏಜೆಂಟ್‌ಗಳನ್ನು ದಿಶಿತ ಗ್ರಾಫ್‌ಗೆ ಸಂಪರ್ಕಿಸುವ ಅವಕಾಶ ನೀಡುತ್ತದೆ. ಇಲ್ಲಿ ನಾವು ಸರಳ ಎರಡು ಹಂತದ ಪೈಪ್ಲೈನ್ ನಿರ್ಮಿಸುತ್ತೇವೆ: **ಟ್ರಾವೆಲ್ ಪ್ಲ್ಯಾನರ್** ಪ್ರವಾಸ ಯೋಜನೆಯನ್ನು ರಚಿಸುತ್ತಾನೆ, ನಂತರ **ಟ್ರಾವೆಲ್ ಕನ್ಸಿಯರ್ಜ್** ಅದನ್ನು ಪರಿಶೀಲಿಸಿ ಸುಧಾರಿಸುತ್ತಾನೆ.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ಕಾರ್ಯಪ್ರವಾಹಕ್ಕೆ ಇನ್ನಷ್ಟು ಏಜೆಂಟ್‌ಗಳನ್ನು ಸೇರಿಸುವುದು

ಬಹು ಏಜೆಂಟ್ ಮಾದರಿಯಲ್ಲಿನ ದೊಡ್ಡ ಪ್ರಯೋಜನಗಳಲ್ಲಿ ಒಂದಾಗಿದ್ದು, ಅದನ್ನು ವಿಸ್ತರಿಸುವುದು 얼마나 ಸುಲಭವೆಂದರೆ. ಕೆಳಗಡೆ ನಾವು **BudgetReviewer** ಏಜೆಂಟ್ ಅನ್ನು ಸೇರಿಸುತ್ತೇವೆ, ಇದು ಯೋಜನೆಯನ್ನು ಪ್ರಯಾಣಿಕರ ಬಜೆಟ್ ವಿರುದ್ಧ ಪರಿಶೀಲಿಸುತ್ತದೆ, ವೆಚ್ಚವನ್ನು ಮಿತಿ ದಾಟುವಂತಹ ಐಟಂಗಳನ್ನು ಸೂಚಿಸುತ್ತದೆ ಮತ್ತು ಹಣ ಉಳಿಸುವ ಪರ್ಯಾಯಗಳನ್ನು ಸೂಚಿಸುತ್ತದೆ. ಕಾರ್ಯಪ್ರವಾಹ ಈಗ ಕ್ರಮವಾಗಿ ಮೂರು ಏಜೆಂಟ್‌ಗಳನ್ನು ચલಿಸುತ್ತದೆ:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು ಕೆಳಕಂಡவற்றನ್ನು ಕಲಿತುಕೊಂಡಿರಿ:

1. **ವಿಶೇಷಿತ ಏಜೆಂಟುಗಳನ್ನು ರಚಿಸುವುದು** — ಪ್ರತಿ ಒಬ್ಬರಿಗೂ ಗಮನಕ್ಕಿದ್ದ ಪಾತ್ರ (ಯೋಜನೆ, ಕಾನ್‌ಸಿಯರ್ಸ್, ಬಜೆಟ್ ಪರಿಶೀಲನೆ).
2. `WorkflowBuilder` ಮತ್ತು `add_edge` ಬಳಸಿ ಏಜೆಂಟುಗಳನ್ನು ಕ್ರಮಬದ್ಧ ಕಾರ್ಯಪ್ರವಾಹದಲ್ಲಿ ಜೋಡಿಸುವುದು.
3. ಬಹು ಏಜೆಂಟ್ ಪೈಪ್ಲೈನಿನಿಂದ ಔಟ್ಪುಟ್‌ನ್ನು ಸ್ಟ್ರೀಮ್ ಮಾಡಿ, ಎಲ್ಲಿ ಯಾವ ಏಜೆಂಟ್ ಮಾತನಾಡುತ್ತಿದೆ ಎಂದು ಅನುಸರಿಸುವುದು.
4. ಈಗಿನ ಏಜೆಂಟ್‌ಗಳನ್ನು ಬದಲಾಯಿಸದೆ ಶ್ರೇಣಿಗೆ ಹೊಸ ಏಜೆಂಟ್‌ಗಳನ್ನು ಸೇರಿಸುವ ಮೂಲಕ ಕಾರ್ಯಪ್ರವಾಹವನ್ನು ವಿಸ್ತರಿಸುವುದು.

ಬಹು ಏಜೆಂಟ್ ವಿನ್ಯಾಸ ಮಾದರಿ ಪ್ರತಿ ಏಜೆಂಟ್ ಅನ್ನು ಸರಳವಾಗಿಡುತ್ತಾ, ಒಬ್ಬ ಏಜೆಂಟ್ ಒಂದೆರಡು ಸಾಧಿಸಬಹುದಾದುದಕ್ಕಿಂತ ಸಮೃದ್ಧ ಮತ್ತು ಸಮಗ್ರ ಪರಿಶೀಲಿತ ಫಲಿತಾಂಶಗಳನ್ನು ನೀಡುತ್ತದೆ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ಡಾಕ್ಯುಮೆಂಟ್ ಅನ್ನು AI ಭಾಷಾಂತರ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಉಪಯೋಗಿಸಿಕೊಂಡು ಭಾಷಾಂತರಿಸಲಾಗಿದೆ. ನಾವು ಶುದ್ಧತೆಗೆ ಪ್ರಯತ್ನಿಸುವುದಾದರೂ, ಸ್ವಯಂಚಾಲಿತ ಭಾಷಾಂತರಗಳಲ್ಲಿ ತಪ್ಪುಗಳು ಅಥವಾ ಅಸೂಕ್ತತೆಗಳು ಇರಬಹುದೆಂದು ದಯವಿಟ್ಟು ಗಮನಿಸಿ. ಮೂಲ ಡಾಕ್ಯುಮೆಂಟ್ ಅದರ ಮೂಲ ಭಾಷೆಯಲ್ಲಿಯೇ ಅಧಿಕೃತ ಸ್ರೋವಾಗಿ ಪರಿಗಣಿಸಲಾಗಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ ವೃತ್ತಿಪರ ಮಾನವ ಭಾಷಾಂತರವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಭಾಷಾಂತರ ಬಳಕೆಯಿಂದ ಉಂಟಾಗುವ ಯಾವುದೇ ದುಂಭಾವನೆಗಳು ಅಥವಾ ತಪ್ಪು ವಿವರಣೆಗಳಿಗೆ ನಾವು ಹೊಣೆಗಾರರಾಗುವುದಿಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
